# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # metadata is an object, do not subscript or iterate over it

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @ids, fields and columns
def print_record_sets_info(dataset):
    print("Available Record Sets and their fields (referenced by `@id`):\n")
    for rs in dataset.metadata.record_sets:
        print(f"RecordSet: {rs.name} (@id: {rs.id})")
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                print(f"  Field: {f.name} (@id: {f.id}) | Column(s): {[col.id for col in getattr(f, 'columns', [])]}")
        print()

print_record_sets_info(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify record set IDs (from overview; fill in actual IDs after running previous cell)
# For this dataset, usual Croissant datasets have one main record set, and its '@id' can be listed using metadata methods.

# List record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record Set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    print(f"Fields (columns): {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick the first record set and try EDA on typical protein fields (e.g., molecular_weight, coverage, peptide_count)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Explore typical numeric fields (List column names)
print("DataFrame columns:", df.columns.tolist())

# Try to select a numeric field (e.g., 'molecular_weight', 'coverage_percent', fallback to any with float/int dtype)
import numpy as np
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if not numeric_fields:
    # Attempt to convert columns with possible numeric names
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print("Detected numeric fields:", numeric_fields)

# Select the first available numeric field
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    raise ValueError('No numeric field found in the record set for EDA.')

# Set a threshold for filtering (median as example)
threshold = df[numeric_field].median() if df[numeric_field].notnull().any() else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (median):")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Try to group by a likely categorical field (e.g. 'description' or similar)
possible_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If there is a second numeric field, show scatterplot
if len(numeric_fields) > 1:
    numeric_field2 = numeric_fields[1]
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=df[numeric_field], y=df[numeric_field2])
    plt.xlabel(numeric_field)
    plt.ylabel(numeric_field2)
    plt.title(f"{numeric_field} vs {numeric_field2}")
    plt.show()

# Boxplot by group (if group field exists and few unique values)
if group_field and df[group_field].nunique() < 20:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have:
- Loaded Croissant metadata and explored record sets using `mlcroissant`
- Loaded data into DataFrames and identified key fields by their `@id`
- Applied basic EDA, filtering by a numeric field, normalizing values, and grouping by categories
- Visualized the data distribution and basic relationships

For further analysis, refine selection of fields by the dataset schema and extend data processing as appropriate for your scientific questions.